# Speaker Segmentation / Diarization
## This notebook outlines the concepts behind segmenting different speakers in the audio file

### Import the libraries

In [2]:
import os, sklearn.cluster
from pyAudioAnalysis.MidTermFeatures import mid_feature_extraction as mT
from pyAudioAnalysis.audioBasicIO import read_audio_file, stereo_to_mono
from pyAudioAnalysis.audioSegmentation import labels_to_segments
import numpy as np
import scipy.io.wavfile as wavfile
import IPython

def normalize_features(features_list):
    """Normalize features to zero mean and unit variance (replaces removed pyAudioAnalysis function)."""
    all_features = np.vstack(features_list)
    MEAN = np.mean(all_features, axis=0)
    STD  = np.std(all_features, axis=0)
    STD[STD == 0] = 1e-10          # avoid division by zero
    normalized = np.vstack([(f - MEAN) / STD for f in features_list])
    return normalized, MEAN, STD

print("Libraries imported successfully")

Libraries imported successfully


### Read signal

In [3]:
fs, x = read_audio_file('Speaker_Diarization_Example.wav')
print(f"Sample rate: {fs} Hz")
print(f"Signal shape: {x.shape}")

Sample rate: 48000 Hz
Signal shape: (3839616, 2)


In [4]:
x = stereo_to_mono(x)
print(f"Signal shape after stereo-to-mono: {x.shape}")
print(f"Duration: {len(x)/fs:.2f} seconds")

Signal shape after stereo-to-mono: (3839616,)
Duration: 79.99 seconds


In [5]:
IPython.display.display(IPython.display.Audio('Speaker_Diarization_Example.wav'))

<IPython.lib.display.Audio object>

### Specify parameters

In [19]:
mt_size, mt_step, st_win, st_step  = 1, 1, 0.05, 0.05

### Extract Mid-term features

In [20]:
[mt_feats, st_feats, _] = mT(x, fs, mt_size * fs, mt_step * fs,
                            round(fs * st_win), round(fs * st_win * 0.5))

### Get normalized segment feature statistics

In [21]:
(mt_feats_norm, MEAN, STD) = normalize_features([mt_feats.T])
mt_feats_norm = mt_feats_norm.T
print(f"Normalized features shape: {mt_feats_norm.shape}")
print(f"MEAN shape: {MEAN.shape}, STD shape: {STD.shape}")

Normalized features shape: (136, 80)
MEAN shape: (136,), STD shape: (136,)


### Perform clustering

In [26]:
n_clusters = 3
x_clusters = [np.array([]) for _ in range(n_clusters)]
k_means = sklearn.cluster.KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
k_means.fit(mt_feats_norm.T)
cls = k_means.labels_
print(f"Clustering complete — labels found: {set(cls)}")
print(f"Cluster distribution: {np.bincount(cls)}")

Clustering complete — labels found: {np.int32(0), np.int32(1), np.int32(2)}
Cluster distribution: [24 31 25]


### Save clusters to concatenated wav files

In [27]:
segs, c = labels_to_segments(cls, mt_step)  # convert flags to segment limits
for sp in range(n_clusters):                
    count_cl = 0
    for i in range(len(c)):     # for each segment in each cluster (>2 secs long)
        if c[i] == sp and segs[i, 1]-segs[i, 0] > 2:
            count_cl += 1
            # get the signal and append it to the cluster's signal (followed by some silence)
            cur_x = x[int(segs[i, 0] * fs): int(segs[i, 1] * fs)]
            x_clusters[sp] = np.append(x_clusters[sp], cur_x)
            x_clusters[sp] = np.append(x_clusters[sp], np.zeros((fs,)))
    # write cluster's signal into a WAV file
    print(f'speaker {sp}: {count_cl} segments {len(x_clusters[sp])/float(fs)} sec total dur')        
    wavfile.write(f'diarization_cluster_{sp}.wav', fs, np.int16(x_clusters[sp]))
    IPython.display.display(IPython.display.Audio(f'diarization_cluster_{sp}.wav'))

speaker 0: 3 segments 22.0 sec total dur


<IPython.lib.display.Audio object>

speaker 1: 2 segments 15.0 sec total dur


<IPython.lib.display.Audio object>

speaker 2: 2 segments 8.0 sec total dur


<IPython.lib.display.Audio object>

### Try some other files with different speakers to group

In [11]:
# Try with Speaker_Diarization_Example_2.wav
print("Processing Speaker_Diarization_Example_2.wav ...")
fs2, x2 = read_audio_file('Speaker_Diarization_Example_2.wav')
x2 = stereo_to_mono(x2)
print(f"Sample rate: {fs2} Hz  |  Duration: {len(x2)/fs2:.2f} s")
IPython.display.display(IPython.display.Audio('Speaker_Diarization_Example_2.wav'))

[mt_feats2, st_feats2, _] = mT(x2, fs2, mt_size * fs2, mt_step * fs2,
                                round(fs2 * st_win), round(fs2 * st_win * 0.5))
(mt_feats2_norm, MEAN2, STD2) = normalize_features([mt_feats2.T])
mt_feats2_norm = mt_feats2_norm.T

k_means2 = sklearn.cluster.KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
k_means2.fit(mt_feats2_norm.T)
cls2 = k_means2.labels_
print(f"Cluster distribution: {np.bincount(cls2)}")

segs2, c2 = labels_to_segments(cls2, mt_step)
x_clusters2 = [np.array([]) for _ in range(n_clusters)]
for sp in range(n_clusters):
    count_cl = 0
    for i in range(len(c2)):
        if c2[i] == sp and segs2[i, 1] - segs2[i, 0] > 2:
            count_cl += 1
            cur_x = x2[int(segs2[i, 0] * fs2): int(segs2[i, 1] * fs2)]
            x_clusters2[sp] = np.append(x_clusters2[sp], cur_x)
            x_clusters2[sp] = np.append(x_clusters2[sp], np.zeros((fs2,)))
    dur = len(x_clusters2[sp]) / float(fs2)
    print(f"speaker {sp}: {count_cl} segments  {dur:.2f} sec total")
    if len(x_clusters2[sp]) > 0:
        out_file = f'diarization2_cluster_{sp}.wav'
        wavfile.write(out_file, fs2, np.int16(x_clusters2[sp]))
        IPython.display.display(IPython.display.Audio(out_file))

Processing Speaker_Diarization_Example_2.wav ...
Sample rate: 48000 Hz  |  Duration: 41.98 s


<IPython.lib.display.Audio object>

Cluster distribution: [201 219]
speaker 0: 3 segments  22.80 sec total


<IPython.lib.display.Audio object>

speaker 1: 3 segments  24.90 sec total


<IPython.lib.display.Audio object>